# Logistic Regression Model

## 1. Imports

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from pathlib import Path
import sys
sys.path.append('../')
from src.utils import save_results

## 2. Load Data

In [2]:
print("Logistic Regression: Loading final pre-processed dataset...")
input_path = Path("../data/processed/final_ml_ready_dataset.csv")
results_path = "../results/model_comparison.csv"

try:
    df = pd.read_csv(input_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: Dataset not found at '{input_path}'. Please run all data preparation scripts first.")

Logistic Regression: Loading final pre-processed dataset...
Dataset loaded successfully. Shape: (2619, 202)


## 3. Define Features (X) and Target (y)

In [3]:
target_column = 'is_fraud'
X = df.drop(columns=[target_column])
y = df[target_column]

## 4. Split Data

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

## 5. Define and Train Model

In [5]:
print("Logistic Regression: Training model...")
model_name = "Logistic Regression"
hyperparams = {'random_state': 42, 'class_weight': 'balanced', 'solver': 'liblinear'}

# Create a pipeline to scale features and then train the model
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(**hyperparams))
])

pipeline.fit(X_train, y_train)
print("Model trained.")

Logistic Regression: Training model...
Model trained.


## 6. Evaluate Model

In [6]:
print("Logistic Regression: Evaluating model...")
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

Logistic Regression: Evaluating model...


## 7. Save Results

In [7]:
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1_score': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_pred_proba)
}

description = f"A linear model for binary classification. Includes StandardScaler. Hyperparameters: {hyperparams}"

save_results(results_path, model_name, description, metrics)

Updated results for 'Logistic Regression' in '..\results\model_comparison.csv'.


## LLM Summary
### Findings
The Logistic Regression model provides a solid linear baseline. Its results show a classic trade-off: a high recall (~0.73) at the expense of precision (~0.68). This indicates the model is effective at identifying a large portion of the actual fraud cases but also produces a fair number of false positives. The use of `StandardScaler` is critical here, as linear models are sensitive to the scale of input features. The ROC AUC score (~0.89) is quite strong for a linear model, suggesting it does a good job of ranking transactions by risk.
### Insights
For a business, this model could be used as a 'wide net' to catch potential fraud for further review. Its high recall makes it valuable for minimizing missed fraud, but the operational cost of reviewing the false positives (lower precision) must be considered. It's a good first-pass filter before sending suspicious cases to a more precise, computationally expensive model or a human analyst.
### Feature Importance Interpretation
Unlike tree-based models, Logistic Regression assigns a 'coefficient' to each feature, indicating its linear relationship with the fraud outcome. We would expect features like `consistency_score` (with a negative coefficient, meaning lower scores increase fraud probability) and `entity_has_crypto` (with a positive coefficient) to have large-magnitude coefficients. This provides a clear, though linear, understanding of which factors contribute most significantly to a transaction being flagged as fraudulent.